# Повторяющиеся идентификаторы

Тематический этап подготовки Европы 2019–2023. Выполняйте ячейки сверху вниз в окружении проекта. Каждый ноутбук соответствует одному шагу; диагностика и применение согласованного изменения разделены. Входы, выходы и порядок описаны в [карте подготовки](README.md). Сохраненные результаты перенесены из выполненного общего ноутбука; сам перенос данных не преобразует.

## Шаг 9 – Повторяющиеся flight_id

Читаем последнюю копию `03_c40_diagnostic_flags.parquet`. Сначала считаем идентификаторы по всему файлу, затем извлекаем все записи повторяющихся значений. Сравниваем UTC-даты, ICAO24, аэропорты, позывные, времена и содержательные поля. Проверяем все пары внутри каждой группы, включая возможные совпадения в один день; примерная голова файла не используется для общих выводов.

В [документации разработчиков traffic](https://traffic-viz.github.io/api_reference/traffic.core.traffic.html#traffic.core.Traffic.assign_id) идентификатор может строиться из позывного и последовательного индекса. [Flight.assign_id](https://traffic-viz.github.io/api_reference/traffic.core.flight.html#traffic.core.Flight.assign_id) описывает шаблон `{self.callsign}_{idx:>03}`. Сходство нашего формата – подсказка, а не доказательство алгоритма автора за 2019–2023 годы. Сохраненный публичный ноутбук автора читает готовый Parquet и не объясняет создание этих идентификаторов.

Повтор flight_id не равен дублю посадки. Отдельно сравниваем все поля рабочей копии без служебного индекса и набор исходных содержательных полей после нормализации. Ключ `landing_time + icao24 + airport` проверен в аудите 01; его уникальность в файле не доказывает идентичность физического полета. Все источники и текущая копия проверяются по SHA256.

In [1]:
# Исследуем повторяющиеся flight_id, не удаляя наблюдения
from pathlib import Path
import runpy
import json
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'europe/data/source/osn23_landings_merged.parquet.gzip').exists())
# Скрипт сравнивает даты, адреса, аэропорты и точные значения внутри групп
module = runpy.run_path(str(ROOT / 'europe/scripts/diagnose_flight_ids.py'))
id_groups, id_categories, id_examples, id_near_pairs = module['main']()
id_journal = json.loads((ROOT / 'europe/reports/preparation/08_flight_id_diagnostics.json').read_text(encoding='utf8'))
# Показываем размеры групп и отличаем коллизии имен от точных дублей
display(id_categories)
display(pd.read_csv(ROOT / 'europe/reports/preparation/08_flight_id_group_sizes.csv'))
print('Повторяющихся идентификаторов:', id_journal['repeated_identifiers'])
print('Записей в этих группах:', id_journal['rows_with_repeated_identifier'])
print('Превышение числа записей над числом идентификаторов:', id_journal['excess_identifier_rows'])
print('Точных дублей рабочей копии в группах:', id_journal['exact_working_duplicate_excess_in_repeated_groups'])
print('Групп с разными адресами:', id_journal['groups_with_multiple_aircraft_addresses'])
print('Групп с разными аэропортами:', id_journal['groups_with_multiple_airports'])
print('Пар одного адреса и аэропорта в одну UTC-дату:', id_journal['same_day_same_address_airport_pairs'])

Проверено строк: 1000000


Проверено строк: 2000000


Проверено строк: 3000000


Проверено строк: 4000000


Проверено строк: 5000000


...

Проверено строк: 7000000


{
  "input": "europe\\data\\working\\03_c40_diagnostic_flags.parquet",
  "input_sha256": "ddf11438bab44137242ffcf4da928418e116efd46defce58dc8a388b67942b6d",
  "source_sha256": "1160bdbea6fa93249a00768452be64047c6d2461bb0843d5ecfc52e6942c7f02",
  "input_unchanged": true,
  "source_unchanged": true,
  "rows_checked": 7984254,
  "rows_changed": 0,
  "rows_deleted": 0,
  "repeated_identifiers": 23889,
  "rows_with_repeated_identifier": 53127,
  "excess_identifier_rows": 29238,
  "go_arounds_in_repeated_identifier_rows": 121,
  "max_rows_per_identifier": 7,
  "exact_working_duplicate_excess_in_repeated_groups": 0,
  "normalized_source_business_duplicate_excess_in_repeated_groups": 0,
  "core_key_duplicate_excess_in_repeated_groups": 0,
  "groups_with_multiple_aircraft_addresses": 22547,
  "groups_with_multiple_airports": 1667,
  "groups_with_multiple_callsigns_including_null": 0,
  "groups_with_at_least_one_go_around": 121,
  "groups_with_mixed_go_around_labels": 121,
  "groups_with_multipl

,category,identifier_groups,rows,go_arounds
0,different_dates_only,23889,53127,121


,rows_per_identifier,identifier_groups,go_arounds
0,2,20069,89
1,3,2594,14
2,4,925,13
3,5,300,5
4,7,1,0


Повторяющихся идентификаторов: 23889
Записей в этих группах: 53127
Превышение числа записей над числом идентификаторов: 29238
Точных дублей рабочей копии в группах: 0
Групп с разными адресами: 22547
Групп с разными аэропортами: 1667
Пар одного адреса и аэропорта в одну UTC-дату: 0


In [2]:
# Разбираем примеры коллизий flight_id с полным контекстом
print('Полные примеры групп повторяющихся идентификаторов:')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(id_examples)
print('Записей с отмеченным уходом в повторяющихся группах:', id_journal['go_arounds_in_repeated_identifier_rows'])
print('Групп со смешанными GoA:', id_journal['groups_with_mixed_go_around_labels'])
# Оцениваем потери целевого класса при гипотетическом удалении по имени
print('Если оставить первую строку каждого flight_id, потеряем отмеченных уходов:',
      id_journal['go_arounds_lost_if_deduplicating_flight_id_keep_first_in_file_order'])
print('Это расчет последствий возможного правила; строки не удалялись.')

Полные примеры групп повторяющихся идентификаторов:


,flight_id,source_row_position,icao24,airport,callsign,landing_date_utc,landing_time,first_attempt_time,GoA,n_attempts,c40_after_landing
8575,ETH500_129,1968662,040030,EIDW,ETH500,2019-10-11,2019-10-11 04:18:08+00:00,2019-10-11 04:18:08+00:00,False,1,False
15794,ETH500_129,3040984,040046,EIDW,ETH500,2020-04-16,2020-04-16 03:55:29+00:00,2020-04-16 03:55:29+00:00,False,1,False
25752,ETH500_129,3905261,040030,EIDW,ETH500,2021-05-17,2021-05-17 04:10:48+00:00,2021-05-17 04:10:48+00:00,False,1,False
35500,ETH500_129,5096302,04002f,EIDW,ETH500,2022-01-23,2022-01-23 04:15:10+00:00,2022-01-23 04:14:53+00:00,False,1,False
51197,ETH500_129,7702038,040032,EIDW,ETH500,2023-04-11,2023-04-11 04:34:23+00:00,2023-04-11 04:34:06+00:00,False,1,False
8710,ETH500_130,1998882,040030,EIDW,ETH500,2019-10-15,2019-10-15 03:31:14+00:00,2019-10-15 03:31:14+00:00,False,1,False
15861,ETH500_130,3046418,040046,EIDW,ETH500,2020-04-25,2020-04-25 03:49:28+00:00,2020-04-25 03:49:26+00:00,False,1,False
25781,ETH500_130,3909981,040030,EIDW,ETH500,2021-05-19,2021-05-19 03:55:36+00:00,2021-05-19 03:55:36+00:00,False,1,False
35558,ETH500_130,5105047,04002f,EIDW,ETH500,2022-01-25,2022-01-25 03:44:25+00:00,2022-01-25 03:44:25+00:00,False,1,False
51305,ETH500_130,7718398,040032,EIDW,ETH500,2023-04-14,2023-04-14 03:46:23+00:00,2023-04-14 03:46:09+00:00,False,1,False


Записей с отмеченным уходом в повторяющихся группах: 121
Групп со смешанными GoA: 121
Если оставить первую строку каждого flight_id, потеряем отмеченных уходов: 71
Это расчет последствий возможного правила; строки не удалялись.


In [3]:
# Проверяем, различает ли UTC-дата строки внутри каждого повторяющегося имени
print('Группы с повтором даты или контекста в один день:')
display(id_groups.loc[~id_groups['date_alone_disambiguates_identifier']])
# Показываем близкие пары и подтверждаем отсутствие изменений базы
display(id_near_pairs)
print('Исходник и рабочая копия неизменны:', id_journal['source_unchanged'], id_journal['input_unchanged'])
print('Изменено / удалено строк:', id_journal['rows_changed'], id_journal['rows_deleted'])

Группы с повтором даты или контекста в один день:


,flight_id,rows,go_arounds,dates,years,addresses,airports,callsigns,first_landing,last_landing,c40_after_landing_rows,landing_span_days,date_alone_disambiguates_identifier,same_day_address_airport_repeat,category


,flight_id,source_row_position_a,icao24_a,airport_a,callsign_a,landing_date_utc_a,landing_time_a,first_attempt_time_a,GoA_a,n_attempts_a,...,first_attempt_time_b,GoA_b,n_attempts_b,same_icao24,same_airport,same_callsign,same_landing_date_utc,same_landing_time,landing_gap_minutes,attempt_to_landing_windows_overlap


Исходник и рабочая копия неизменны: True True
Изменено / удалено строк: 0 0


## Итог шага 9

23 889 значений flight_id повторяются в 53 127 записях. Все записи внутри каждого такого идентификатора относятся к разным UTC-датам. В 22 547 группах различаются ICAO24; в 1 667 – аэропорты. Совпадение имени не связывает их в один физический полет. Среди этих записей нет точных содержательных дублей или повторов ключа посадки.

121 группа содержит ровно одну запись GoA=True и другие записи GoA=False. Удаление по flight_id может уничтожить реальные наблюдения и изменить состав целевого класса. Идентификатор нельзя использовать как единственный ключ присоединения меток, попыток или траекторий; иначе возможны связи между разными датами и бортами. Совпадение `flight_id + UTC-дата посадки` различает записи текущего файла, но не гарантирует уникальность будущих данных и не заменяет проверку траекторного сопоставления, особенно около полуночи.

Решение: все строки сохраняем, исходный flight_id не переписываем. При подготовке связей используем проверенный составной ключ наблюдения и контекст времени, адреса и аэропорта. Отдельный идентификатор записи можно добавить следующим согласованным правилом; он будет служебным, а не признаком модели. Идентификатор физического полета потребует иной проверки.

Подробный [разбор повторов](../../reports/preparation/08_flight_id_review.md) содержит ограничения и первичные источники. Следующий диагностический этап – записи с большим числом попыток; они не считаются ошибочными или учебными автоматически.